# XAI-MeteoFormer — resubmission runs (GPU track)

**Run with `Save & Run All (Commit)`, not interactively.**

Order, and roughly what each costs on a T4:

| # | Cell | Time |
|---|---|---|
| 1 | Config | — |
| 2 | Setup: clone repo + pinned TSLib | 2 min |
| 3 | Environment capture (Minor #8) | 1 min |
| 4 | Resume from a previous output | 1 min |
| 5 | **P0-1a memory + batch probe** | ~15 min |
| 6 | **P0-1b Crossformer 256x1024, Jena, 5 seeds** | ~5-7 h |
| 7 | **P0-1c Crossformer 256x512, Jena, 5 seeds** | ~4-5 h |
| 8 | **P0-4 no_entropy: 2 Jena + 5 Beijing** | ~1.3 h |
| 9 | Export: what to download | 1 min |
| 10-12 | **Second pass** — P0-2, P1-1, P1-2 | see below |

Cells 6+7+8 together are ~11-13 h, i.e. **more than one 12-hour session**.
Every cell resumes on the existing key, so a session that dies costs one run,
not the batch. Recommended: session 1 = cells 5+6+8, session 2 = cell 7.

Cells 10-12 are the **second pass** — do not run them until the Crossformer
result has been looked at.


In [ ]:
# ------------------------------- CONFIG -------------------------------- #
REPO_URL   = "https://github.com/shapokok/xai-meteoformer.git"   # <-- yours
CODE_INPUT = None          # or "/kaggle/input/<your-code-dataset>"
DATA_INPUT = "/kaggle/input/xai-meteoformer-processed"           # *_X.npy + *_meta.json

# Pinned so the baselines are reproducible. Bump deliberately, never silently:
# a different TSLib commit can change a baseline's forward pass.
TSLIB_COMMIT = "4e938a1767106324dd753b2a44832bf870a0252e"

# Stop launching new runs after this many hours so the output still saves.
DEADLINE_HOURS = 11.0

# Batch size for the Crossformer width runs. Cell 5 tells you whether 128 is
# safe; leave at 64 until it has said so.
CROSSFORMER_BATCH = 64


In [ ]:
import os, subprocess, sys, time, shutil, pathlib
T_START = time.time()

WORK  = pathlib.Path("/kaggle/working")
REPO  = WORK / "xai-meteoformer"
TSLIB = WORK / "Time-Series-Library"

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=check)

if CODE_INPUT:
    if REPO.exists():
        shutil.rmtree(REPO)
    shutil.copytree(CODE_INPUT, REPO)
elif not REPO.exists():
    sh(f"git clone --depth 1 {REPO_URL} {REPO}")

if not TSLIB.exists():
    # Shallow fetch of ONE pinned commit, not of whatever HEAD happens to be.
    TSLIB.mkdir(parents=True)
    sh(f"cd {TSLIB} && git init -q && "
       f"git remote add origin https://github.com/thuml/Time-Series-Library.git && "
       f"git fetch -q --depth 1 origin {TSLIB_COMMIT} && "
       f"git checkout -q FETCH_HEAD")
sh(f"cd {TSLIB} && git rev-parse HEAD")

sh("pip install -q reformer_pytorch einops")

os.environ["TSLIB_PATH"] = str(TSLIB)
os.chdir(REPO)
OUT = REPO / "outputs"
OUT.mkdir(exist_ok=True)
print("cwd:", os.getcwd())
print("data:", sorted(os.listdir(DATA_INPUT)))


In [ ]:
# ---- environment capture (Reviewer 1, Minor #8) ------------------------ #
import json, platform, torch

freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True).stdout
(OUT / "requirements_frozen.txt").write_text(freeze)

tslib_sha = subprocess.run(f"cd {TSLIB} && git rev-parse HEAD", shell=True,
                           capture_output=True, text=True).stdout.strip()
env = {
    "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE",
    "gpu_total_MiB": (torch.cuda.get_device_properties(0).total_memory // 2**20
                      if torch.cuda.is_available() else 0),
    "tslib_commit": tslib_sha, "tslib_commit_pinned": TSLIB_COMMIT,
}
(OUT / "environment.json").write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))
assert tslib_sha == TSLIB_COMMIT, f"TSLib drifted: {tslib_sha} != {TSLIB_COMMIT}"
print(f"\npip freeze: {len(freeze.splitlines())} packages")


In [ ]:
# ---- resume from a previous run of this notebook ----------------------- #
prev = None
for p in pathlib.Path("/kaggle/input").glob("*/outputs"):
    if (p / "results.csv").exists():
        prev = p
        break
if prev:
    print("resuming from", prev)
    shutil.copy(prev / "results.csv", OUT / "results.csv")
    for sub in ("checkpoints", "predictions"):
        if (prev / sub).exists():
            shutil.copytree(prev / sub, OUT / sub, dirs_exist_ok=True)
    import pandas as pd
    print(len(pd.read_csv(OUT / "results.csv")), "runs already present")
else:
    print("no previous output found; starting fresh")


In [ ]:
# ===================== P0-1a  memory + batch probe ===================== #
# Confirms the local estimate (2384 MiB at batch 64) on the real T4 and tells
# you whether batch 128 changes convergence. One epoch each, nothing kept.
import torch, subprocess, sys, re, json

EST_MB = {64: 2384, 128: 4606}          # from analysis/crossformer_memory_probe.py
probe = {}

for bs in (64, 128):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    cmd = [sys.executable, "src/train.py",
           "--dataset", "jena", "--model_name", "Crossformer",
           "--bl_d_model", "256", "--bl_d_ff", "1024",
           "--batch_size", str(bs), "--seed", "0",
           "--epochs", "1", "--patience", "1", "--force",
           "--processed_dir", DATA_INPUT, "--tslib_path", str(TSLIB),
           "--out_dir", str(WORK / "probe"),
           "--results_csv", str(WORK / "probe" / "probe.csv")]
    t = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    peak = torch.cuda.max_memory_allocated() / 2**20
    m = re.findall(r"val ([0-9.]+)", r.stdout)
    val = float(m[-1]) if m else float("nan")
    probe[bs] = {"peak_MiB": peak, "val_loss": val,
                 "epoch_s": round(time.time() - t, 1), "rc": r.returncode}
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
    print(f"batch={bs}: peak {peak:7.0f} MiB (estimate {EST_MB[bs]})  "
          f"val {val:.4f}  {probe[bs]['epoch_s']} s/epoch  rc={r.returncode}",
          flush=True)

total = torch.cuda.get_device_properties(0).total_memory / 2**20
print(f"\nGPU total {total:.0f} MiB")
for bs, p in probe.items():
    ratio = p["peak_MiB"] / EST_MB[bs]
    if ratio > 1.5 or p["peak_MiB"] > 0.8 * total:
        print("\n" + "!" * 70)
        print(f"!! batch={bs}: peak {p['peak_MiB']:.0f} MiB is {ratio:.1f}x the "
              f"estimate {EST_MB[bs]} MiB ({100*p['peak_MiB']/total:.0f}% of the card).")
        print("!! DO NOT run the main batch at this size. Drop to batch 64, or")
        print("!! to d_ff=512, and re-run this probe.")
        print("!" * 70 + "\n")

v64, v128 = probe[64]["val_loss"], probe[128]["val_loss"]
if v64 == v64 and v128 == v128:
    rel = abs(v128 - v64) / max(v64, 1e-9)
    print(f"\nval loss after 1 epoch: batch64 {v64:.4f}  batch128 {v128:.4f}  "
          f"({100*rel:.1f}% apart)")
    print("-> batch 128 is safe, set CROSSFORMER_BATCH = 128 and re-run cell 1"
          if rel < 0.05 else
          "-> batch 128 changes convergence; KEEP CROSSFORMER_BATCH = 64")
json.dump(probe, open(OUT / "crossformer_probe.json", "w"), indent=2)


In [ ]:
# ============ P0-1b  Crossformer d_model=256 d_ff=1024, Jena =========== #
# 10.6 M params. Our model is 1.89 M -- report parameter counts alongside.
import subprocess, sys, time

def run(args, label):
    if (time.time() - T_START) / 3600 > DEADLINE_HOURS:
        print(f"== deadline reached, stopping before {label} =="); return False
    t = time.time()
    r = subprocess.run([sys.executable, "src/train.py",
                        "--processed_dir", DATA_INPUT,
                        "--tslib_path", str(TSLIB),
                        "--out_dir", str(OUT),
                        "--results_csv", str(OUT / "results.csv")] + args,
                       capture_output=True, text=True)
    tail = [l for l in r.stdout.splitlines() if l.strip()][-6:]
    print(f"[{label}] rc={r.returncode} {(time.time()-t)/60:.1f} min")
    for l in tail: print("   ", l)
    if r.returncode != 0: print(r.stderr[-2000:])
    return True

for seed in range(5):
    run(["--dataset", "jena", "--model_name", "Crossformer",
         "--bl_d_model", "256", "--bl_d_ff", "1024",
         "--batch_size", str(CROSSFORMER_BATCH), "--seed", str(seed)],
        f"Crossformer 256x1024 jena s{seed}")


In [ ]:
# ============ P0-1c  Crossformer d_model=256 d_ff=512, Jena ============ #
# 7.98 M params -- the middle capacity point. Run in a SECOND session if the
# first one is out of time.
for seed in range(5):
    run(["--dataset", "jena", "--model_name", "Crossformer",
         "--bl_d_model", "256", "--bl_d_ff", "512",
         "--batch_size", str(CROSSFORMER_BATCH), "--seed", str(seed)],
        f"Crossformer 256x512 jena s{seed}")


In [ ]:
# ================= P0-4  no_entropy to 5 seeds, both datasets ========== #
# Currently 3 seeds on Jena and none on Beijing. Needed for Major #7 so the
# XAI comparison against the headline variant is on equal budgets.
for seed in (3, 4):
    run(["--dataset", "jena", "--model_name", "XAI-MeteoFormer",
         "--ablation", "no_entropy", "--seed", str(seed)],
        f"no_entropy jena s{seed}")

for seed in range(5):
    run(["--dataset", "beijing_aotizhongxin", "--model_name", "XAI-MeteoFormer",
         "--ablation", "no_entropy", "--seed", str(seed)],
        f"no_entropy beijing s{seed}")


In [ ]:
# ========================= export: what to download ==================== #
import pandas as pd, shutil, pathlib

res = OUT / "results.csv"
if res.exists():
    d = pd.read_csv(res)
    new = d[(d.get("width", "default") != "default") |
            (d.ablation == "no_entropy")]
    print(f"{len(d)} rows total, {len(new)} from this work:")
    cols = [c for c in ["model","dataset","ablation","seed","width","params",
                        "MAE","RMSE","R2"] if c in d.columns]
    print(new[cols].to_string(index=False))

# Ship only the NEW artefacts, so the download stays small.
ship = WORK / "to_download"
shutil.rmtree(ship, ignore_errors=True)
(ship / "checkpoints").mkdir(parents=True)
(ship / "predictions").mkdir(parents=True)
n = 0
for p in (OUT / "checkpoints").glob("*.pt"):
    if "_w256x" in p.name or "no_entropy" in p.name:
        shutil.copy(p, ship / "checkpoints" / p.name); n += 1
for p in (OUT / "predictions").glob("*.npy"):
    if "_w256x" in p.name or "no_entropy" in p.name:
        shutil.copy(p, ship / "predictions" / p.name)
if res.exists():
    shutil.copy(res, ship / "results_new_kaggle.csv")
for f in ("environment.json", "requirements_frozen.txt", "crossformer_probe.json"):
    if (OUT / f).exists():
        shutil.copy(OUT / f, ship / f)
print(f"\n{n} new checkpoints staged in {ship}")

print("""
DOWNLOAD /kaggle/working/to_download AND UNPACK LOCALLY AS:

  to_download/results_new_kaggle.csv  ->  <repo>/results_new_kaggle.csv
  to_download/checkpoints/*.pt        ->  <repo>/checkpoints/
  to_download/predictions/*.npy       ->  <repo>/predictions/
  to_download/environment.json        ->  <repo>/analysis/
  to_download/requirements_frozen.txt ->  <repo>/analysis/
  to_download/crossformer_probe.json  ->  <repo>/analysis/

Then, locally:

  python analysis/repair_results_csv.py      # picks up results_new*.csv by glob
  python analysis/crossformer_fullwidth.py   # writes analysis/crossformer_fullwidth.md

No analysis script needs editing: repair_results_csv.py globs results_new*.csv,
and the checkpoint/prediction naming already carries the width tag.
""")


---

# SECOND PASS — do not run until the Crossformer result has been reviewed

The three cells below are P0-2, P1-1 and P1-2. They are independent of each
other and of everything above.

| Cell | What | Time |
|---|---|---|
| P0-2 | window sweep, seq_len 24/48/192, 3 seeds, both datasets | ~4 h |
| P1-1 | normalization variants for 5 baselines, 5 seeds, both datasets | ~4 h |
| P1-2 | ablation to 5 seeds on both datasets | ~6 h |


In [ ]:
# ===== P0-2  window-length sweep (Reviewer 1, Major #6) ================ #
# Tests the mechanism found in analysis/occlusion_time.md: occlusion says the
# model uses only the last ~16 h of the 96 h window, so short windows should
# lose little. Headline configuration, 3 seeds.
# NOTE: seq_len is part of neither the resume key nor the checkpoint tag, so
# each seq_len is written to its OWN results csv and out_dir to avoid
# colliding with the seq_len=96 runs.
for L in (24, 48, 192):
    for ds in ("jena", "beijing_aotizhongxin"):
        for seed in range(3):
            sub = OUT / f"seqlen{L}"
            sub.mkdir(exist_ok=True)
            if (time.time() - T_START) / 3600 > DEADLINE_HOURS:
                print("== deadline reached =="); break
            t = time.time()
            r = subprocess.run([sys.executable, "src/train.py",
                                "--dataset", ds, "--model_name", "XAI-MeteoFormer",
                                "--ablation", "no_revin", "--seed", str(seed),
                                "--seq_len", str(L),
                                "--processed_dir", DATA_INPUT,
                                "--tslib_path", str(TSLIB),
                                "--out_dir", str(sub),
                                "--results_csv", str(sub / "results.csv")],
                               capture_output=True, text=True)
            print(f"[seq_len={L} {ds} s{seed}] rc={r.returncode} "
                  f"{(time.time()-t)/60:.1f} min", flush=True)
            if r.returncode != 0: print(r.stderr[-1500:])


In [ ]:
# ===== P1-1  normalization symmetry (5 baselines) ====================== #
# --norm_variant on  : adds the SAME RevIN our model uses to the baselines
#                      whose forward pass has none
# --norm_variant off : removes our RevIN from LSTM, i.e. the textbook baseline
# Crossformer is deliberately excluded here -- it is handled by P0-1.
for ds in ("jena", "beijing_aotizhongxin"):
    for m in ("DLinear", "Transformer", "Informer", "Autoformer"):
        for seed in range(5):
            run(["--dataset", ds, "--model_name", m,
                 "--norm_variant", "on", "--seed", str(seed)],
                f"{m} norm=on {ds} s{seed}")
    for seed in range(5):
        run(["--dataset", ds, "--model_name", "LSTM",
             "--norm_variant", "off", "--seed", str(seed)],
            f"LSTM norm=off {ds} s{seed}")


In [ ]:
# ===== P1-2  top every ablation up to 5 seeds, both datasets =========== #
ABL = ["no_multiscale", "no_var_attn", "no_temp_attn",
       "no_fusion", "no_entropy", "no_cls"]
for ds in ("jena", "beijing_aotizhongxin"):
    for abl in ABL:
        for seed in range(5):
            run(["--dataset", ds, "--model_name", "XAI-MeteoFormer",
                 "--ablation", abl, "--seed", str(seed)],
                f"{abl} {ds} s{seed}")
